#Tokenization Techniques in NLP
Week 2 NLP Pipeline
PBA/ Genap 2025/ Irmasari Hafidz
irma@its.ac.id

## Install Dependencies


In [18]:
!pip install nltk
!pip install spacy sacremoses sentencepiece
!pip install transformers

! python3 -m spacy download en_core_web_sm

from spacy.lang.en import English
nlp = English()


   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   ---------- ----------------------------- 2.6/10.1 MB 17.1 MB/s eta 0:00:01
   ---------------- ----------------------- 4.2/10.1 MB 13.5 MB/s eta 0:00:01
   -------------------------------- ------- 8.1/10.1 MB 14.0 MB/s eta 0:00:01
   ---------------------------------------- 10.1/10.1 MB 14.4 MB/s  0:00:00
   ---------------------------------------- 0.0/625.2 kB ? eta -:--:--
   ---------------------------------------- 625.2/625.2 kB 15.1 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ------------------------------- -------- 2.9/3.7 MB 14.6 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 13.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   -------------------------------------- - 2.6/2.7 MB 13.1 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 11.0 MB/s  0:00:00

  Attempting uninstall: regex


  You can safely remove it manually.
Python was not found; run without arguments to install from the Microsoft Store, or disable this shortcut from Settings > Apps > Advanced app settings > App execution aliases.


## Import Required Libraries

In [1]:
import nltk
import spacy
import re
from nltk.tokenize import word_tokenize, sent_tokenize, RegexpTokenizer
from nltk.tokenize.treebank import TreebankWordTokenizer
from sacremoses import MosesTokenizer
import sentencepiece as spm
import pandas as pd

In [2]:
df = pd.read_csv("data.csv")
print(df.head())

                                             content    label
0  I like the game and is have funny legend and s...  Positif
1  I just retrieved my acc, great customer servic...  Positif
2                                          good game  Positif
3                                          it's best  Positif
4                                               good  Positif


### White Space Tokenization
*   one of the simplest tokenization techniques as it uses whitespace within the string as the delimiter of words.
*   Wherever the white space is, it will split the data at that point
*   Using **Python built** .split() in or **Spacy**

```
# sentence.split()
```




In [3]:
df["split_tokens"] = df["content"].apply(lambda x: x.split())
df["split_tokens"]

0      [I, like, the, game, and, is, have, funny, leg...
1      [I, just, retrieved, my, acc,, great, customer...
2                                           [good, game]
3                                           [it's, best]
4                                                 [good]
                             ...                        
169    [i, love, mlbb, but, I'll, hope, I, got, luckk...
170    [An, appeal, should, be, validated, by, Moonto...
171    [Very, very, awesome, and, fun, to, play,, but...
172    [The, game, improved, a, lot, in, all, areas,,...
173    [my, first, MOBA, game,, it's, not, that, bad,...
Name: split_tokens, Length: 174, dtype: object

In [5]:
from nltk.tokenize import WhitespaceTokenizer

# Create a reference variable for Class WhitespaceTokenizer
wtk = WhitespaceTokenizer()

#use tokenize method
df["whitespace_tokens"] = df["content"].apply(lambda x: wtk.tokenize(x))
df["whitespace_tokens"]

0      [I, like, the, game, and, is, have, funny, leg...
1      [I, just, retrieved, my, acc,, great, customer...
2                                           [good, game]
3                                           [it's, best]
4                                                 [good]
                             ...                        
169    [i, love, mlbb, but, I'll, hope, I, got, luckk...
170    [An, appeal, should, be, validated, by, Moonto...
171    [Very, very, awesome, and, fun, to, play,, but...
172    [The, game, improved, a, lot, in, all, areas,,...
173    [my, first, MOBA, game,, it's, not, that, bad,...
Name: whitespace_tokens, Length: 174, dtype: object

#### **Tokenization** using Spacy and NLTK

Word level Tokenization

In [8]:
import spacy

nlp = spacy.load('en_core_web_sm')

def spacy_token(text):
  doc= nlp(text)
  return [token.text for token in doc]

df["spacy_tokens"] = df["content"].apply(spacy_token)
df[["content", "spacy_tokens"]].head()

,content,spacy_tokens
0,I like the game and is have funny legend and s...,"[I, like, the, game, and, is, have, funny, leg..."
1,"I just retrieved my acc, great customer servic...","[I, just, retrieved, my, acc, ,, great, custom..."
2,good game,"[good, game]"
3,it's best,"[it, 's, best]"
4,good,[good]


In [9]:
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')
df["nltk_tokens"] = df["content"].apply(lambda x: word_tokenize(x))
df["nltk_tokens"].head()

[nltk_data] Downloading package punkt_tab to C:\Users\HP
[nltk_data]     840\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


0    [I, like, the, game, and, is, have, funny, leg...
1    [I, just, retrieved, my, acc, ,, great, custom...
2                                         [good, game]
3                                       [it, 's, best]
4                                               [good]
Name: nltk_tokens, dtype: object

In [15]:
#import sent_tokenize from nltk library
from nltk import sent_tokenize
def tokenize_process(text):
    # Pecah jadi kalimat, lalu tiap kalimat dipecah jadi kata
    return [word_tokenize(sent) for sent in sent_tokenize(text)]

df['sent_tokenize'] = df['content'].apply(tokenize_process)
df['sent_tokenize'][1]

[['I',
  'just',
  'retrieved',
  'my',
  'acc',
  ',',
  'great',
  'customer',
  'services',
  '.'],
 ['It', 'really', 'helped', 'me', 'alot']]

Character Tokenization

In [ ]:
text = "Jerman"
characters = list(text)
print("Character Tokens:", characters)

Character Tokens: ['J', 'e', 'r', 'm', 'a', 'n']


Subword Tokenization

*   breaks down words into smaller units called subwords.
*   used to handle unknown or rare words by breaking them into small known words.



In [20]:
from transformers import BertTokenizer

bert = BertTokenizer.from_pretrained('bert-base-uncased')

df["bert_tokens"] = df["content"].apply(lambda x: bert.tokenize(x))
df["bert_tokens"].head()

0    [i, like, the, game, and, is, have, funny, leg...
1    [i, just, retrieved, my, acc, ,, great, custom...
2                                         [good, game]
3                                     [it, ', s, best]
4                                               [good]
Name: bert_tokens, dtype: object

In [ ]:
import nltk
import spacy
import re
from nltk.tokenize import word_tokenize, sent_tokenize, RegexpTokenizer
from nltk.tokenize.treebank import TreebankWordTokenizer
from sacremoses import MosesTokenizer
import sentencepiece as spm

# White Space Tokenization
def whitespace_tokenization(text):
    return text.split()

# Regular Expression Tokenizer
def regex_tokenization(text):
    tokenizer = RegexpTokenizer(r'\w+')
    return tokenizer.tokenize(text)

# Penn Treebank Tokenization
def penn_treebank_tokenization(text):
    tokenizer = TreebankWordTokenizer()
    return tokenizer.tokenize(text)

# SpaCy Tokenization
nlp = spacy.load("en_core_web_sm")
def spacy_tokenization(text):
    doc = nlp(text)
    return [token.text for token in doc]

# Moses Tokenization
moses_tokenizer = MosesTokenizer()
def moses_tokenization(text):
    return moses_tokenizer.tokenize(text)


In [21]:
df.head()

,content,label,split_tokens,whitespace_tokens,spacy_tokens,nltk_tokens,tokenized_content,sent_tokenize,bert_tokens
0,I like the game and is have funny legend and s...,Positif,"[I, like, the, game, and, is, have, funny, leg...","[I, like, the, game, and, is, have, funny, leg...","[I, like, the, game, and, is, have, funny, leg...","[I, like, the, game, and, is, have, funny, leg...","[[I, like, the, game, and, is, have, funny, le...","[[I, like, the, game, and, is, have, funny, le...","[i, like, the, game, and, is, have, funny, leg..."
1,"I just retrieved my acc, great customer servic...",Positif,"[I, just, retrieved, my, acc,, great, customer...","[I, just, retrieved, my, acc,, great, customer...","[I, just, retrieved, my, acc, ,, great, custom...","[I, just, retrieved, my, acc, ,, great, custom...","[[I, just, retrieved, my, acc, ,, great, custo...","[[I, just, retrieved, my, acc, ,, great, custo...","[i, just, retrieved, my, acc, ,, great, custom..."
2,good game,Positif,"[good, game]","[good, game]","[good, game]","[good, game]","[[good, game]]","[[good, game]]","[good, game]"
3,it's best,Positif,"[it's, best]","[it's, best]","[it, 's, best]","[it, 's, best]","[[it, 's, best]]","[[it, 's, best]]","[it, ', s, best]"
4,good,Positif,[good],[good],[good],[good],[[good]],[[good]],[good]
